# Assignment 2 - classification and evaluation

**Foundations of Machine Learning (Demo), E1234 - individual, 20% of the final mark.**
Due Tuesday 3 November 2026, 23:59 (Europe/Berlin). Push to `main` in this repository;
that push is your submission.

Implement the seven functions below using **only the Python standard library** (`math` is
fine). Then use them, in section 9, to choose and defend a decision threshold.

> **Read this before you start.** The grader converts this notebook to a script and
> imports it, so **every top-level cell runs at grading time**. A cell that raises stops
> the import and costs you all eight automated marks. That is why the exploratory cells
> below are wrapped in `try/except NotImplementedError` - keep them that way, and keep any
> experiments of your own inside the same guard.

## 0. Data

In [ ]:
# The data. Twenty-four fictional loan applications: two standardised features and a
# binary outcome (1 = defaulted). Small enough to check any calculation by hand.
import math

X = [
    [1.2, -0.4], [0.3, 0.9], [-0.8, 1.6], [2.1, -1.1], [-1.4, 0.2], [0.7, 0.5],
    [-0.2, -0.9], [1.8, 0.3], [-1.1, 1.2], [0.9, -1.4], [-0.5, 0.7], [1.5, 1.1],
    [-1.7, -0.3], [0.4, 1.8], [1.1, -0.7], [-0.9, -1.2], [2.3, 0.6], [-0.3, 1.4],
    [0.6, -0.2], [-1.2, 0.9], [1.4, -1.6], [-0.7, 0.4], [0.2, 1.3], [1.9, -0.9],
]
y = [0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0]

print(f"n = {len(y)}, positives = {sum(y)}, base rate = {sum(y) / len(y):.3f}")

## 1. `sigmoid`

The squashing function that turns a linear score into a probability. The only subtlety is
numerical: `math.exp` overflows for arguments above about 710.

In [ ]:
def sigmoid(z):
    """The logistic function, computed without overflowing for large |z|."""
    if z >= 0:
        return 1.0 / (1.0 + math.exp(-z))
    e = math.exp(z)                     # exp of a NEGATIVE number never overflows
    return e / (1.0 + e)

## 2. `log_loss`

The loss we minimise. Each term is the negative log probability the model assigned to what
actually happened, so a confident wrong answer is punished hard.

In [ ]:
def log_loss(y_true, probs, eps=1e-12):
    """Mean negative log-likelihood, with the probabilities clipped away from 0 and 1."""
    total = 0.0
    for actual, p in zip(y_true, probs):
        p = min(max(p, eps), 1.0 - eps)
        total -= actual * math.log(p) + (1 - actual) * math.log(1.0 - p)
    return total / len(y_true)

## 3. `predict_proba`

Coefficients in, probabilities out. Keep the convention from Assignment 1: the intercept is
`beta[0]` and `X` carries no column of ones.

In [ ]:
def predict_proba(X, beta):
    """Intercept plus the dot product of the row with the slopes, squashed."""
    return [sigmoid(beta[0] + sum(b * v for b, v in zip(beta[1:], row))) for row in X]

## 4. `fit_logistic`

Session 3's descent loop with session 4's gradient. Note how little changes: the gradient is
still "project the error back onto each feature", with `p` in place of `y_hat`.

In [ ]:
def fit_logistic(X, y, alpha=0.5, n_iter=2000):
    """Batch gradient descent on the log-loss. grad = (1/n) X'(p - y)."""
    n = len(y)
    design = [[1.0] + list(row) for row in X]      # leading column of ones
    beta = [0.0] * len(design[0])
    for _ in range(n_iter):
        probs = [sigmoid(sum(b * v for b, v in zip(beta, row))) for row in design]
        grad = [0.0] * len(beta)
        for row, p, actual in zip(design, probs, y):
            err = p - actual
            for j, value in enumerate(row):
                grad[j] += err * value / n
        beta = [b - alpha * g for b, g in zip(beta, grad)]
    return beta

## 5. `confusion_counts`

Four numbers, and everything in section 6 is a ratio of them. Get the order right:
`(tp, fp, tn, fn)`.

In [ ]:
def confusion_counts(y_true, y_pred):
    """(tp, fp, tn, fn) for 0/1 labels."""
    tp = sum(1 for a, p in zip(y_true, y_pred) if a == 1 and p == 1)
    fp = sum(1 for a, p in zip(y_true, y_pred) if a == 0 and p == 1)
    tn = sum(1 for a, p in zip(y_true, y_pred) if a == 0 and p == 0)
    fn = sum(1 for a, p in zip(y_true, y_pred) if a == 1 and p == 0)
    return tp, fp, tn, fn

## 6. `precision_recall_f1`

Of those flagged, how many were right (precision); of those that mattered, how many were
caught (recall); and their harmonic mean.

In [ ]:
def precision_recall_f1(y_true, y_pred):
    """(precision, recall, f1), with 0.0 wherever a denominator vanishes."""
    tp, fp, _tn, fn = confusion_counts(y_true, y_pred)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1

## 7. `choose_threshold`

The model is fixed by section 4. This function makes the *decision*, and it needs the costs
to do so - which is the whole argument of session 4.

In [ ]:
def choose_threshold(y_true, probs, cost_fn, cost_fp, grid=None):
    """Minimise cost_fn*FN + cost_fp*FP over the grid; ties go to the lower threshold."""
    if grid is None:
        grid = [round(0.05 * k, 2) for k in range(1, 20)]
    best_threshold, best_cost = None, None
    for threshold in sorted(grid):
        labels = [1 if p >= threshold else 0 for p in probs]
        _tp, fp, _tn, fn = confusion_counts(y_true, labels)
        cost = cost_fn * fn + cost_fp * fp
        if best_cost is None or cost < best_cost:   # strict <: keeps the LOWER threshold
            best_threshold, best_cost = threshold, cost
    return best_threshold, best_cost

## 8. Put it together

In [ ]:
# A self-check you can run as you go. Wrapped so that an unimplemented function cannot
# break the import - see the warning at the top of this notebook.
try:
    print(f"sigmoid(0)     = {sigmoid(0):.4f}   (expect 0.5)")
    print(f"sigmoid(-1000) = {sigmoid(-1000):.4g}  (must not raise)")
    print(f"log_loss([1, 0], [0.5, 0.5]) = {log_loss([1, 0], [0.5, 0.5]):.4f}   "
          f"(expect {math.log(2):.4f})")
    print(f"confusion_counts([1,1,0,0], [1,0,1,0]) = "
          f"{confusion_counts([1, 1, 0, 0], [1, 0, 1, 0])}   (expect (1, 1, 1, 1))")
except NotImplementedError as exc:
    print("not implemented yet:", exc)

In [ ]:
# Fit, evaluate, and choose a threshold.
try:
    beta = fit_logistic(X, y)
    probs = predict_proba(X, beta)
    print("coefficients (intercept first):", [round(b, 3) for b in beta])
    print(f"training log-loss: {log_loss(y, probs):.4f}   "
          f"(all-zero coefficients would give {math.log(2):.4f})")

    print("\n thresh  precision  recall      F1   cost(FN=5, FP=1)")
    for t in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
        labels = [1 if p >= t else 0 for p in probs]
        precision, recall, f1 = precision_recall_f1(y, labels)
        _tp, fp, _tn, fn = confusion_counts(y, labels)
        print(f"  {t:.2f}     {precision:6.3f}  {recall:6.3f}  {f1:6.3f}   {5 * fn + fp:4d}")

    print("\ncost-minimising threshold (FN 5x FP):", choose_threshold(y, probs, 5, 1))
    print("cost-minimising threshold (FP 5x FN):", choose_threshold(y, probs, 1, 5))
except NotImplementedError as exc:
    print("not implemented yet:", exc)

## 9. Model write-up (the marking reference)

**(a) The cost ratio.** A default that is not caught costs the lender the outstanding
principal, while a needless rejection costs a modest amount of goodwill and one staff review
- so a false negative is worth roughly five false positives here. That ratio is a policy
choice, not a statistical one, and it should be set by whoever owns the loss, not by the
analyst.

**(b) The threshold.** At `cost_fn = 5, cost_fp = 1` the cost-minimising threshold sits well
below 0.5, raising recall and lowering precision: more applicants are flagged for review, and
fewer defaults slip through. Compared with the 0.5 default, borderline applicants who would
have been approved are now reviewed.

**(c) What this does not tell you.** Twenty-four rows, all used for fitting: there is no
held-out estimate at all, so every number above is optimistic and none has an interval. It
also says nothing about whether the two features are available at decision time, or whether
error rates are equal across applicant groups - sessions 9 and 10.

*Full credit needs a stated ratio, the threshold it implies with its precision/recall, and a
limitation that is specific rather than generic.*